Paths and reusable definitions were refactored. Stored outputs below are historical, not a fresh execution. See `docs/WORKFLOW.md` at the repository root before rerunning; several stages overwrite results or append Optuna trials.


In [ ]:
# Locate this checkout from the repository root or any notebook subdirectory.
from pathlib import Path
import sys

_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "src" / "steam_recommender" / "paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
sys.path.insert(0, str(_root / "src"))
from steam_recommender import paths as _paths
if Path(_paths.__file__).resolve() != (_root / "src/steam_recommender/paths.py").resolve():
    raise RuntimeError("A different checkout is cached; restart the kernel.")
from steam_recommender.paths import project_root, project_path


In [ ]:
# =========================
# FINAL TEST EVALUATION — ALS + POPULAR BASELINE
# =========================

import os
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# =========================
# PATHS
# =========================


data_dir = os.path.join(str(project_path("data/processed")))
optuna_results_dir = os.path.join(str(project_path("results/optuna")))
final_results_dir = os.path.join(str(project_path("results/final")))

os.makedirs(final_results_dir, exist_ok=True)

# =========================
# IMPORT FROM PIPELINE
# =========================

from steam_recommender.als import ALSPipeline
from steam_recommender.confidence import linear_signal, log_signal, log_saturated_signal, make_log_inside_beta_signal
from steam_recommender.metrics import precision_at_k, recall_at_k, ndcg_at_k, coverage_at_k, delta_gap


# =========================
# FILE PATHS
# =========================

best_configs_path = os.path.join(
    optuna_results_dir,
    "best_als_configs.csv"
)

test_metrics_path = os.path.join(
    final_results_dir,
    "test_metrics_by_seed.csv"
)

per_user_ndcg_path = os.path.join(
    final_results_dir,
    "per_user_ndcg.csv"
)

In [2]:
# =========================
# LOAD DATA
# =========================

train_val_df = pd.read_csv(
    os.path.join(data_dir, "users_items_filtered_train_val.csv")
)

test_df = pd.read_csv(
    os.path.join(data_dir, "users_items_filtered_test.csv")
)

best_configs = pd.read_csv(best_configs_path)


# Typy jako stringi — spójnie z pipeline
for df in [train_val_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

display(best_configs)

print("train_val_df:", train_val_df.shape)
print("test_df:", test_df.shape)

C:\Users\misie\AppData\Local\Temp\ipykernel_19632\809147695.py:5: DtypeWarning: Columns (0: user_id) have mixed types. Specify dtype option on import or set low_memory=False.
  train_val_df = pd.read_csv(


,model,function,factors,iterations,alpha,beta,regularization,validation_ndcg,study_name,n_trials
0,ALS-linear,linear,80,30,9.948124e-07,NaN,0.003588,0.248913,als_linear,150
1,ALS-log,log,24,30,5.713901e-04,NaN,0.130920,0.261320,als_log,150
2,ALS-log-saturation,log_saturation,24,30,2.439038e-02,NaN,0.169714,0.261400,als_log_saturation,150
3,ALS-log-inside-beta,log_inside_beta,48,30,3.891845e-01,108.283984,0.010565,0.275349,als_log_inside_beta,150


train_val_df: (4591379, 4)
test_df: (541231, 4)


In [3]:
# =========================
# HELPERS
# =========================

from steam_recommender.confidence import get_four_variant_signal as get_signal_fn


from steam_recommender.baselines import evaluate_popular_test as evaluate_popular_baseline


function_labels = {
    "popular": "popularność",
    "linear": "liniowa",
    "log": "logarytmiczna",
    "log_saturation": "logarytmiczna z saturacją",
    "log_inside_beta": "logarytmiczna z przesunięciem β",
}


In [ ]:
# =========================
# FINAL EVALUATION SETTINGS
# =========================

K = 10
seeds = [0, 1, 2, 3, 4]

num_threads = 8

metric_rows = []
per_user_rows = []

In [5]:
# =========================
# FINAL TEST EVALUATION — ALS MODELS
# =========================

for _, config in best_configs.iterrows():
    model_name = config["model"]
    function_name = config["function"]

    factors = int(config["factors"])
    iterations = int(config["iterations"])
    alpha = float(config["alpha"])
    beta = config["beta"] if "beta" in config and not pd.isna(config["beta"]) else None
    regularization = float(config["regularization"])

    signal_fn = get_signal_fn(function_name, beta=beta)

    print("\n=========================")
    print(f"Model: {model_name}")
    print(
        f"function={function_name}, "
        f"factors={factors}, iterations={iterations}, "
        f"alpha={alpha}, beta={beta}, reg={regularization}"
    )

    for seed in tqdm(seeds, desc=model_name):
        pipeline = ALSPipeline(
            factors=factors,
            iterations=iterations,
            alpha=alpha,
            regularization=regularization,
            random_state=seed,
            signal_strength_fn=signal_fn,
            num_threads=num_threads,
        )

        results, recommendations, user_ndcg = pipeline.refit_full_and_test(
            train_df=train_val_df,
            test_df=test_df,
            K=K,
            return_per_user=True,
            verbose=False,
        )

        metric_row = {
            "model": model_name,
            "function": function_name,
            "function_label": function_labels.get(function_name, function_name),
            "seed": seed,
            "factors": factors,
            "iterations": iterations,
            "alpha": alpha,
            "beta": beta,
            "regularization": regularization,
        }

        metric_row.update(results)
        metric_rows.append(metric_row)

        for user_id, ndcg_value in user_ndcg.items():
            per_user_rows.append({
                "model": model_name,
                "function": function_name,
                "function_label": function_labels.get(function_name, function_name),
                "seed": seed,
                "user_id": user_id,
                f"ndcg@{K}": ndcg_value,
            })


test_metrics_df = pd.DataFrame(metric_rows)
per_user_ndcg_df = pd.DataFrame(per_user_rows)

display(test_metrics_df.head())
display(per_user_ndcg_df.head())


Model: ALS-linear
function=linear, factors=80, iterations=30, alpha=9.94812441289902e-07, beta=None, reg=0.0035878904678559


ALS-linear:   0%|          | 0/5 [00:00<?, ?it/s]

d:\Anaconda\envs\recommender\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

,model,function,function_label,seed,factors,iterations,alpha,beta,regularization,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,users_evaluated
0,ALS-linear,linear,liniowa,0,80,30,9.948124e-07,NaN,0.003588,0.184005,0.264033,0.330002,0.117960,-0.212504,0.789018,62944
1,ALS-linear,linear,liniowa,1,80,30,9.948124e-07,NaN,0.003588,0.183722,0.263528,0.329067,0.117308,-0.213088,0.780653,62944
2,ALS-linear,linear,liniowa,2,80,30,9.948124e-07,NaN,0.003588,0.184113,0.264080,0.329935,0.118613,-0.210822,0.790994,62944
3,ALS-linear,linear,liniowa,3,80,30,9.948124e-07,NaN,0.003588,0.184410,0.264631,0.330450,0.116873,-0.209020,0.776920,62944
4,ALS-linear,linear,liniowa,4,80,30,9.948124e-07,NaN,0.003588,0.184251,0.264578,0.330196,0.118178,-0.210772,0.789579,62944


,model,function,function_label,seed,user_id,ndcg@10
0,ALS-linear,linear,liniowa,0,--000--,0.000000
1,ALS-linear,linear,liniowa,0,--ace--,0.098039
2,ALS-linear,linear,liniowa,0,--ionex--,0.000000
3,ALS-linear,linear,liniowa,0,-2SV-vuLB-Kg,0.000000
4,ALS-linear,linear,liniowa,0,-404PageNotFound-,0.630043


In [6]:
# =========================
# FINAL TEST EVALUATION — POPULAR BASELINE
# =========================

print("\n=========================")
print("Popular baseline")

popular_results, popular_user_ndcg = evaluate_popular_baseline(
    train_df=train_val_df,
    test_df=test_df,
    K=K,
    return_per_user=True,
)

popular_metric_row = {
    "model": "Popular",
    "function": "popular",
    "function_label": function_labels["popular"],
    "seed": np.nan,
    "factors": np.nan,
    "iterations": np.nan,
    "alpha": np.nan,
    "beta": np.nan,
    "regularization": np.nan,
}

popular_metric_row.update(popular_results)

# Dopisujemy Popular do metryk
test_metrics_df = pd.concat(
    [
        test_metrics_df,
        pd.DataFrame([popular_metric_row])
    ],
    ignore_index=True
)

# Per-user NDCG dla Popular też zapisujemy, może się przydać jako baseline.
popular_per_user_rows = []

for user_id, ndcg_value in popular_user_ndcg.items():
    popular_per_user_rows.append({
        "model": "Popular",
        "function": "popular",
        "function_label": function_labels["popular"],
        "seed": np.nan,
        "user_id": user_id,
        f"ndcg@{K}": ndcg_value,
    })

per_user_ndcg_df = pd.concat(
    [
        per_user_ndcg_df,
        pd.DataFrame(popular_per_user_rows)
    ],
    ignore_index=True
)

display(test_metrics_df.tail())
display(per_user_ndcg_df.tail())


Popular baseline


,model,function,function_label,seed,factors,iterations,alpha,beta,regularization,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,users_evaluated
16,ALS-log-inside-beta,log_inside_beta,logarytmiczna z przesunięciem β,1.0,48.0,30.0,0.389185,108.283984,0.010565,0.195791,0.333897,0.351698,0.131115,0.422487,0.728584,62944
17,ALS-log-inside-beta,log_inside_beta,logarytmiczna z przesunięciem β,2.0,48.0,30.0,0.389185,108.283984,0.010565,0.195417,0.333203,0.351134,0.130681,0.417238,0.738857,62944
18,ALS-log-inside-beta,log_inside_beta,logarytmiczna z przesunięciem β,3.0,48.0,30.0,0.389185,108.283984,0.010565,0.195618,0.333966,0.351897,0.131007,0.419228,0.704153,62944
19,ALS-log-inside-beta,log_inside_beta,logarytmiczna z przesunięciem β,4.0,48.0,30.0,0.389185,108.283984,0.010565,0.195523,0.333901,0.351655,0.132855,0.416728,0.734175,62944
20,Popular,popular,popularność,NaN,NaN,NaN,NaN,NaN,NaN,0.096519,0.178252,0.185480,0.008698,1.423258,NaN,62944


,model,function,function_label,seed,user_id,ndcg@10
1321819,Popular,popular,popularność,NaN,zzonci,0.000000
1321820,Popular,popular,popularność,NaN,zzoptimuszz,0.000000
1321821,Popular,popular,popularność,NaN,zzydrax,0.386853
1321822,Popular,popular,popularność,NaN,zzyfo,0.235046
1321823,Popular,popular,popularność,NaN,zzzmidmiss,0.274876


In [7]:
# =========================
# SAVE FINAL TEST RESULTS
# =========================

test_metrics_df.to_csv(test_metrics_path, index=False)
per_user_ndcg_df.to_csv(per_user_ndcg_path, index=False)

print(f"Saved test metrics to: {test_metrics_path}")
print(f"Saved per-user NDCG to: {per_user_ndcg_path}")

print("\nShapes:")
print("test_metrics_df:", test_metrics_df.shape)
print("per_user_ndcg_df:", per_user_ndcg_df.shape)

Saved test metrics to: D:\Studiowe\Licencjat\modele\final_results\test_metrics_by_seed.csv
Saved per-user NDCG to: D:\Studiowe\Licencjat\modele\final_results\per_user_ndcg.csv

Shapes:
test_metrics_df: (21, 16)
per_user_ndcg_df: (1321824, 6)


In [8]:
# =========================
# SUMMARY: TEST METRICS MEAN ± STD
# =========================

metric_cols = [
    f"precision@{K}",
    f"recall@{K}",
    f"ndcg@{K}",
    f"coverage@{K}",
    f"delta_gap@{K}",
    f"diversity@{K}",
    "users_evaluated",
]

summary_rows = []

for (model_name, function_name), group in test_metrics_df.groupby(["model", "function"]):
    row = {
        "model": model_name,
        "function": function_name,
        "function_label": group["function_label"].iloc[0],
        "n_runs": group["seed"].notna().sum(),
    }

    for metric in metric_cols:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=1)

    summary_rows.append(row)

test_summary_df = pd.DataFrame(summary_rows)

function_order = [
    "popular",
    "linear",
    "log",
    "log_saturation",
    "log_inside_beta",
]

test_summary_df["function_order"] = test_summary_df["function"].map(
    {name: i for i, name in enumerate(function_order)}
)

test_summary_df = (
    test_summary_df
    .sort_values("function_order")
    .drop(columns=["function_order"])
    .reset_index(drop=True)
)

display(test_summary_df)

,model,function,function_label,n_runs,precision@10_mean,precision@10_std,recall@10_mean,recall@10_std,ndcg@10_mean,ndcg@10_std,coverage@10_mean,coverage@10_std,delta_gap@10_mean,delta_gap@10_std,diversity@10_mean,diversity@10_std,users_evaluated_mean,users_evaluated_std
0,Popular,popular,popularność,0,0.096519,NaN,0.178252,NaN,0.185480,NaN,0.008698,NaN,1.423258,NaN,NaN,NaN,62944.0,NaN
1,ALS-linear,linear,liniowa,5,0.184100,0.000260,0.264170,0.000452,0.329930,0.000523,0.117786,0.000694,-0.211241,0.001607,0.785433,0.006251,62944.0,0.0
2,ALS-log,log,logarytmiczna,5,0.180967,0.000103,0.301856,0.000173,0.335613,0.000295,0.076169,0.000250,0.543323,0.000807,0.596301,0.015880,62944.0,0.0
3,ALS-log-saturation,log_saturation,logarytmiczna z saturacją,5,0.180868,0.000166,0.301907,0.000302,0.335418,0.000484,0.076104,0.000172,0.544328,0.000893,0.594833,0.015152,62944.0,0.0
4,ALS-log-inside-beta,log_inside_beta,logarytmiczna z przesunięciem β,5,0.195639,0.000180,0.333795,0.000334,0.351690,0.000352,0.131333,0.000866,0.419129,0.002309,0.731420,0.017398,62944.0,0.0


In [9]:
# =========================
# PRETTY TABLE: MEAN ± STD
# =========================

pretty_rows = []

for _, row in test_summary_df.iterrows():
    pretty_row = {
        "model": row["model"],
        "function": row["function_label"],
    }

    for metric in metric_cols:
        mean = row[f"{metric}_mean"]
        std = row[f"{metric}_std"]

        if pd.isna(mean):
            pretty_row[metric] = "NA"
        elif row["function"] == "popular":
            # Popular ma pojedynczy wynik, więc bez std
            if metric == "users_evaluated":
                pretty_row[metric] = f"{mean:.0f}"
            else:
                pretty_row[metric] = f"{mean:.4f}"
        else:
            if metric == "users_evaluated":
                pretty_row[metric] = f"{mean:.0f} ± {std:.0f}"
            else:
                pretty_row[metric] = f"{mean:.4f} ± {std:.4f}"

    pretty_rows.append(pretty_row)

pretty_test_summary = pd.DataFrame(pretty_rows)

display(pretty_test_summary)

,model,function,precision@10,recall@10,ndcg@10,coverage@10,delta_gap@10,diversity@10,users_evaluated
0,Popular,popularność,0.0965,0.1783,0.1855,0.0087,1.4233,NA,62944
1,ALS-linear,liniowa,0.1841 ± 0.0003,0.2642 ± 0.0005,0.3299 ± 0.0005,0.1178 ± 0.0007,-0.2112 ± 0.0016,0.7854 ± 0.0063,62944 ± 0
2,ALS-log,logarytmiczna,0.1810 ± 0.0001,0.3019 ± 0.0002,0.3356 ± 0.0003,0.0762 ± 0.0003,0.5433 ± 0.0008,0.5963 ± 0.0159,62944 ± 0
3,ALS-log-saturation,logarytmiczna z saturacją,0.1809 ± 0.0002,0.3019 ± 0.0003,0.3354 ± 0.0005,0.0761 ± 0.0002,0.5443 ± 0.0009,0.5948 ± 0.0152,62944 ± 0
4,ALS-log-inside-beta,logarytmiczna z przesunięciem β,0.1956 ± 0.0002,0.3338 ± 0.0003,0.3517 ± 0.0004,0.1313 ± 0.0009,0.4191 ± 0.0023,0.7314 ± 0.0174,62944 ± 0
